# Vertical integration — a complete walkthrough

**Vertical integration** takes several modalities measured **in the same cell** (here CITE-seq: RNA + surface protein) and learns one joint representation. Cells are already matched, so the task is fusing modalities, not aligning cells.

**Reference dataset:** `D11` (2,864 cells) — **14 methods** run on it end-to-end.

Everything below uses the same three calls, whatever the scenario:

```python
mtb.scan(dataset)                    # what can I run?
res = mtb.run_all(dataset, category) # run it, with metrics
res.plot()                           # one figure
```

In [ ]:
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", None)  # show `reason` in full - paths are long
import multibench as mtb

RESULTS = Path("results")          # stored results, so the comparisons reproduce
print("multibench", mtb.__version__)

In [ ]:
DATASET  = "D11"
CATEGORY = "vertical"
FAST_METHOD = "Matilda"        # used for the quick live demos below

## 0. Which scenario is my data, and how do I lay it out?

Two calls answer what every new user hits first. `category` is a required argument
everywhere, so start by seeing the legal values:

In [ ]:
mtb.list_categories()

And this prints exactly which filenames to use — including how **several
batches** are named (numbered files in one flat directory):

In [ ]:
print(mtb.describe_layout(CATEGORY))

## 1. What can I run on this data?

`scan` inspects the dataset and reports every method that can run — and, for the
rest, exactly why not. Nothing is executed, so this is safe and instant.

In [ ]:
avail = mtb.scan(DATASET, category=CATEGORY)
avail[avail.runnable][["method", "modalities", "env", "output_kind",
                       "n_tunable", "runtime_tier", "observed_worst_sec"]]

Methods that are *not* runnable here come with a reason rather than a silent absence:

In [ ]:
not_ok = avail[~avail.runnable][["method", "modalities", "reason"]]
not_ok.head(5) if len(not_ok) else "(everything in this category runs on this dataset)"


## 2. What can I tune?

`params_for` reports the parameters a method accepts. `defaults` are what the
wrapper passes; `tunable` is what the **upstream script** exposes on its command
line. An empty `tunable` means that method hardcodes its hyperparameters — it
cannot be tuned without editing it, which this project never does.

In [ ]:
rows = []
for m in avail[avail.runnable]["method"]:
    p = mtb.params_for(m, CATEGORY, None if avail[avail.method==m].iloc[0]["modalities"]=="(data_dir)"
                       else avail[avail.method==m].iloc[0]["modalities"].split("+"))
    rows.append({"method": m, "n_tunable": len(p["tunable"]),
                 "defaults": p["defaults"],
                 "example": ", ".join(sorted(p["tunable"])[:4]) or "(hardcoded upstream)"})
pd.DataFrame(rows).sort_values("n_tunable", ascending=False).reset_index(drop=True)

## 3. Run one method

Matilda trains in ~1 min at 5 epochs. Tuning happens through `params=` — the same parameters `params_for`
just listed.

In [ ]:
res = mtb.run_all(DATASET, CATEGORY,
                  methods=["Matilda"], params={"Matilda": {"epochs": 5}},
                  out_dir="/tmp/tutorial_vertical")
res

In [ ]:
res.summary

The metrics are already computed — `run_all` runs the method, picks the correct
label order, and evaluates in one step.

In [ ]:
res.plot()

## 4. Run everything

One call runs every applicable method and evaluates each:

```python
res = mtb.run_all(DATASET, CATEGORY, out_dir="/tmp/vertical_all",
                  timeout=4*3600,        # a hung method is recorded, not fatal
                  skip_existing=True)    # resume instead of repeating hours
print(res.failures)                      # ALWAYS check: failures are recorded, not raised
res.plot()
```

Next morning, re-plot **without re-running anything** (`run_all` saved it for you):

```python
res = mtb.load_batch("/tmp/vertical_all")
res.plot().savefig("compare.png")
```

> WARNING `skip_existing` reuses a method's existing output FILE. Do not combine it
> with a changed `params=` (you would silently get the old result, so `run_all`
> refuses it), and after a hard kill delete that method's directory, since a
> half-written file would be reused as if it had succeeded.

That takes from minutes to hours depending on the scenario, so here we load the
stored results of exactly that sweep.

In [ ]:
summary = pd.read_csv(RESULTS / "summary_D11.csv")
summary

### What a failure looks like

`.failures` is the table to check after every sweep, because a method that dies is
**recorded, not raised** — the sweep keeps going. Here is a genuine failure: a
method given an impossible hyperparameter. Note the sweep still returns normally
and the error is in the table.

In [ ]:
broken = mtb.run_all(DATASET, CATEGORY, methods=[FAST_METHOD],
                     params={FAST_METHOD: {"epochs": -5}},   # nonsense on purpose
                     out_dir="/tmp/tut_fail", verbose=False)
print("failures:", len(broken.failures))
broken.failures

By contrast, if **nothing at all** can run — a mistyped dataset name, say —
that is not a per-method failure, it means the request itself was wrong, so
`run_all` raises instead of handing back an empty result that reads as success:

In [ ]:
try:
    mtb.run_all("NO_SUCH_DATASET", CATEGORY, out_dir="/tmp/tut_none")
except ValueError as e:
    print("ValueError:", str(e)[:160])

## 5. The figure

In [ ]:
long = pd.read_csv(RESULTS / "long_all_D11.csv")
fig = mtb.plot.bubble(long, title="vertical integration — D11")
fig.set_dpi(110)
display(fig)

> The bubble chart encodes **radius = rank** and **fill = value normalised within
> the plotted set** — both relative to what you plotted. Read the table beside it;
> with few methods a tiny gap can look decisive.

## 6. Using your OWN dataset

`describe_layout` above already printed the exact filenames for this scenario —
**use those, not the role names.** They are not always the same: the `atac_gas`
role lives in `atac.h5`, and `atac_peak` lives in `peak.h5`.

```python
print(mtb.describe_layout(CATEGORY))   # the authoritative filename list
```

Then the same three calls work unchanged:

```python
mtb.scan("MYDATA", category="vertical")                     # confirm it is picked up
res = mtb.run_all("MYDATA", "vertical", out_dir="out/")     # run everything
res.plot()
```

If `scan` says a method is not runnable, the `reason` column names the missing
file — fix that and re-scan. Modality files are HDF5 with the matrix under
`matrix/data`; `mtb.io.to_canonical` converts other layouts.